# Case 2 — Seq2Seq + AttentionDịch máy **Anh → Việt** trên IWSLT'15 — LSTM encoder-decoder + Luong attention (nền tảng của `tensorflow/nmt`)> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.>> Toàn bộ notebook mất khoảng **3–5 giờ** trên T4 miễn phí.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smiimport torchassert torch.cuda.is_available(), "Chưa bật GPU! Runtime -> Change runtime type -> T4 GPU"print(f"\nGPU     : {torch.cuda.get_device_name(0)}")print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")print(f"PyTorch : {torch.__version__}")

## 2. Lấy code vào ColabChọn **một** trong hai cách. Đã đẩy project lên GitHub thì dùng cách A; chưa cóthì nén thư mục project thành `.zip` rồi dùng cách B.

In [ ]:
# --- Cách A: clone từ GitHub ---# !git clone https://github.com/<user>/<repo>.git machine_translate# --- Cách B: upload file zip của project ---import os, pathlib, zipfileif not os.path.exists('machine_translate') and not os.path.exists('common'):    from google.colab import files    print("Chọn file zip của project...")    up = files.upload()    with zipfile.ZipFile(list(up)[0]) as z:        z.extractall('.')    if not os.path.exists('machine_translate/common'):        for p in pathlib.Path('.').rglob('common/engine.py'):            os.rename(str(p.parent.parent), 'machine_translate')            breakif os.path.exists('machine_translate'):    %cd machine_translate!ls

## 3. Cài thư việnColab đã có sẵn PyTorch; chỉ cần thêm hai package nhẹ.

In [ ]:
!pip install -q sentencepiece 'sacrebleu>=2.4'print("xong")

## 4. Tải dữ liệu IWSLT'15 En-ViScript tự verify đủ số câu (133,317 / 1,553 / 1,268) và dừng ngay nếu lệch.

In [ ]:
!bash scripts/download_data.sh

## 5. Train tokenizer dùng chung**Chỉ chạy một lần** và phải chạy trước cả hai case — cả hai đều nạp đúng bộtokenizer này, đó là điều kiện để so sánh công bằng. Nếu bạn đã chạy notebookcủa case kia trong cùng phiên Colab thì bỏ qua ô này.

In [ ]:
!python scripts/prepare.py

## 6. Kiểm tra trước khi trainHai bước này rẻ và bắt được lỗi trước khi bạn đốt hàng giờ GPU.- `check_amp.py` — chạy cả hai model dưới fp16. Lỗi lệch kiểu fp16/fp32 vô hình  trên CPU nhưng làm hỏng ngay lần train đầu trên GPU.- `sanity_check.py` — bắt model học thuộc 120 câu; cài đặt đúng phải đạt BLEU > 80.

In [ ]:
!python scripts/check_amp.py

In [ ]:
!python scripts/sanity_check.py --model seq2seq

## 7. Train Seq2Seq + AttentionCấu hình mặc định (`translate_seq2seq/config.py`) bám benchmark En-Vi của`tensorflow/nmt`: encoder 1 lớp bi-LSTM 512, decoder 2 lớp LSTM 512, Luongattention `general` có scale, **input feeding**, dropout 0.2, clip gradient 5.0.**Case này chậm hơn Case 1 đáng kể** — ước tính 8–15 phút/epoch trên T4. Nguyênnhân không phải số tham số (20.0M so với 39.7M của Transformer) mà là inputfeeding: nó buộc decoder chạy tuần tự từng timestep. Chênh lệch này chính là thứbảng so sánh cuối cùng đo.**Colab free thường ngắt phiên sau ~4 giờ, tức nhiều khả năng bạn sẽ bị ngắt ởcase này.** Hai cách xử lý:- Chạy lại ô này với `--resume true` để train tiếp từ `runs/seq2seq/last.pt`.- Hoặc giảm ngay từ đầu: `--epochs 15`.Dù bị ngắt, checkpoint tốt nhất vẫn nằm ở `runs/seq2seq/best.pt` và vẫn đánh giá được.

In [ ]:
!python translate_seq2seq/train.py# Train tiếp sau khi bị ngắt:# !python translate_seq2seq/train.py --resume true# Tái hiện đúng cấu hình gốc tensorflow/nmt (SGD lr=1.0):# !python translate_seq2seq/train.py --optimizer sgd --lr 1.0 --epochs 12

## 8. Đánh giá với beam size khác`train.py` đã chấm sẵn greedy và beam mặc định. Ô này để thử beam khác hoặc xemthêm câu dịch mẫu mà không phải train lại.

In [ ]:
!python evaluate.py --model seq2seq --beam 10 --show 8

## 9. Dịch thử câu bất kỳ

In [ ]:
!python evaluate.py --model seq2seq --beam 10 \    --text "I want to talk about the science behind climate change ."

## 10. So sánh hai caseÔ này chỉ ra kết quả đầy đủ khi **cả hai** case đã train xong trong cùng phiênColab (hoặc bạn đã copy `runs/` của case kia vào).

In [ ]:
!python compare.py --plots

In [ ]:
from IPython.display import Image, Markdown, displayimport osif os.path.exists('runs/comparison.png'):    display(Image('runs/comparison.png'))if os.path.exists('runs/COMPARISON.md'):    display(Markdown(open('runs/COMPARISON.md').read()))

## 11. Tải kết quả về máy`runs/` gồm `benchmark.json` (mọi số đo), `history.csv` (learning curve), câudịch trên tst2013 và checkpoint. Tải về trước khi phiên Colab hết hạn.

In [ ]:
!zip -qr runs.zip runs -x '*/last.pt'from google.colab import filesfiles.download('runs.zip')